### Connection MYSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
import json

### Dosen Tetap Program Studi (DTPS)

In [ ]:

def parse_details(x):
  if pd.isna(x):
    return {}
  try:
    data = json.loads(x)
    for key in ['serdos', 'jabfung', 'gelar_depan', 'gelar_belakang']:
      data.pop(key, None)
    return data
  except Exception:
    return {}

In [ ]:
jenjangPendidikanQuery = text("""
SELECT code, name FROM jenjang_pendidikan
""")

jenjangPendidikanDf = pd.read_sql(jenjangPendidikanQuery, siakadConnection)

In [ ]:
dtpsQuery = text("""
SELECT
    dosen.name AS nama_dosen,
    homebase.name AS homebase,
    semesters.year_of_study AS tahun_ajaran,
    kelas_prodi.name AS kelas_prodi,
    dosen.details
FROM dosen_list AS dosen

JOIN prodi AS homebase ON
    dosen.prodi_uuid = homebase.uuid

JOIN dosen_kelas ON
    dosen_kelas.dosen_uuid = dosen.uuid

JOIN kelas ON
    dosen_kelas.kelas_id = kelas.id

JOIN semesters ON
    kelas.semester_code = semesters.code

JOIN prodi AS kelas_prodi ON
    kelas.prodi_uuid = kelas_prodi.uuid

WHERE
    active = 1 AND
    semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 )

ORDER BY semesters.year_of_study DESC
""")

dtpsDf = pd.read_sql(dtpsQuery, siakadEngine);
details = dtpsDf['details'].apply(parse_details)
details = pd.json_normalize(details)

dtpsDf = pd.concat(
    [dtpsDf.drop(columns=['details']), details], axis=1
)

dtpsDf['pendidikan_terakhir'] = dtpsDf['pendidikan_terakhir'].astype(str)
jenjangPendidikanDf['code'] = jenjangPendidikanDf['code'].astype(str)

dtpsDf = dtpsDf.merge(
    jenjangPendidikanDf, 
    left_on='pendidikan_terakhir', 
    right_on='code', 
    how='left'
)

dtpsDf = dtpsDf.drop(columns=['code'])
dtpsDf = dtpsDf.drop(columns=['pendidikan_terakhir'])
dtpsDf = dtpsDf.rename(columns={'name': 'pendidikan_terakhir'});

In [ ]:
total_data = len(dtpsDf)
print(f"Total data mahasiswa: {total_data}")
dtpsDf